In [1]:
# Weather Status Extraction over Pre-TRACON Grid Cells
# Date: 2023-12-22 00:00 to 00:55 UTC

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, Point
import imageio.v2 as imageio
import requests
from PIL import Image
from io import BytesIO

# --- Grid Setup (copied from user-provided code) ---
dtw_lat, dtw_lon = 42.2125, -83.3534
radius_deg = 3
grid_step = 0.7

star_fixes = {
    "BONZZ": (-82.7972, 41.7483), "CRAKN": (-82.9405, 41.6730), "CUUGR": (-83.0975, 42.3643),
    "FERRL": (-82.6093, 42.4165), "GRAYT": (-83.6020, 42.9150), "HANBL": (-84.1773, 41.7375),
    "HAYLL": (-84.2975, 41.9662), "HTROD": (-83.3442, 42.0278), "KKISS": (-83.7620, 42.5443),
    "KLYNK": (-82.9888, 41.8793), "LAYKS": (-83.5498, 42.8532), "LECTR": (-84.0217, 41.9183),
    "RKCTY": (-83.9603, 42.6869), "VCTRZ": (-84.0670, 41.9878)
}
ordered_fix_names = ["CRAKN", "BONZZ", "FERRL", "LAYKS", "GRAYT", "RKCTY", "HAYLL", "HANBL"]
from shapely.geometry import box
tracon_polygon = Polygon([star_fixes[name] for name in ordered_fix_names])
theta = np.linspace(0, 2 * np.pi, 400)
circle_coords = [(dtw_lon + radius_deg * np.cos(t), dtw_lat + radius_deg * np.sin(t)) for t in theta]
circle_polygon = Polygon(circle_coords)
minx, miny, maxx, maxy = circle_polygon.bounds

grid_cells = []
clipped_cells = []

for lon in np.arange(minx, maxx, grid_step):
    for lat in np.arange(miny, maxy, grid_step):
        cell = box(lon, lat, lon + grid_step, lat + grid_step)
        if circle_polygon.intersects(cell) and not tracon_polygon.contains(cell):
            clipped = cell.intersection(circle_polygon)
            if not clipped.is_empty:
                grid_cells.append(cell)
                clipped_cells.append(clipped)

# --- Function to get dBZ values from image and .wld file ---
def get_geotransform(wld_text):
    vals = list(map(float, wld_text.strip().splitlines()))
    return vals  # xres, 0, 0, -yres, topleft_x, topleft_y

def latlon_from_pixel(i, j, geot):
    xres, _, _, yres, topleft_x, topleft_y = geot
    lon = topleft_x + j * xres
    lat = topleft_y + i * yres
    return lat, lon

def pixel_to_dbz(pixel_val):
    # Map 0–255 to -32 to 90 dBZ (0.5 dBZ steps)
    return -32 + pixel_val * 0.5

# --- Main loop to download, decode and analyze ---
results = []
from datetime import datetime, timedelta

base_time = datetime(2023, 12, 22, 0, 0)
for t_idx in range(12):  # 00:00 to 00:55 UTC
    dt = base_time + timedelta(minutes=5 * t_idx)
    dstr = dt.strftime("%Y%m%d%H%M")
    date_url = dt.strftime("%Y/%m/%d")
    base_url = f"https://mesonet.agron.iastate.edu/archive/data/{date_url}/GIS/uscomp/"
    img_url = base_url + f"n0q_{dstr}.png"
    wld_url = base_url + f"n0q_{dstr}.wld"

    try:
        img_data = Image.open(BytesIO(requests.get(img_url).content)).convert('L')
        wld_text = requests.get(wld_url).text
        geot = get_geotransform(wld_text)
    except:
        print(f"Failed to load {dstr}")
        continue

    img_array = np.array(img_data)
    nrows, ncols = img_array.shape
    pixel_lats = np.empty((nrows, ncols))
    pixel_lons = np.empty((nrows, ncols))

    for i in range(nrows):
        for j in range(ncols):
            lat, lon = latlon_from_pixel(i, j, geot)
            pixel_lats[i, j] = lat
            pixel_lons[i, j] = lon

    points = np.array([pixel_lons.flatten(), pixel_lats.flatten()]).T
    dbz_values = pixel_to_dbz(img_array.flatten())

    for grid_id, poly in enumerate(clipped_cells):
        mask = np.array([poly.contains(Point(lon, lat)) for lon, lat in points])
        values = dbz_values[mask]
        values = values[values >= 0]  # filter only meaningful reflectivity

        if len(values) > 0:
            mean_dbz = values.mean()
            max_dbz = values.max()
            rain_flag = mean_dbz > 15
        else:
            mean_dbz = 0.0
            max_dbz = 0.0
            rain_flag = False

        results.append({
            'time': dt,
            'grid_id': grid_id,
            'mean_dbz': round(mean_dbz, 1),
            'max_dbz': round(max_dbz, 1),
            'rain_flag': rain_flag
        })

# --- Save result ---
df = pd.DataFrame(results)
df.to_csv("weather_grid_20231222_0000_0055.csv", index=False)
df.head()

KeyboardInterrupt: 